In [0]:

# ============================================
# Rearc Data Quest
# Day 1 - BLS Source Inspection
# ============================================

# Step 1: Define Volume path

VOLUME_PATH = "/Volumes/workspace/rearc/rearc_landing"

BLS_DATA_PATH = f"{VOLUME_PATH}/raw/bls/pr.data.1.AllData"

# Step 2: Read BLS data

bls_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "false")
    .csv(BLS_DATA_PATH)
)

# Step 3: Display first 10 records

display(bls_df.limit(10))

# Step 4: Print schema

bls_df.printSchema()

# Step 5: Count records

print("Total records:", bls_df.count())


In [0]:
from pyspark.sql import functions as F

print("Total records:", bls_df.count())

print("Distinct periods:")
display(
    bls_df.groupBy("period")
          .count()
          .orderBy("period")
)

print("Year range:")
bls_df.select(
    F.min("year").alias("min_year"),
    F.max("year").alias("max_year")
).show()

print("Null values:")
bls_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in bls_df.columns
]).show()

In [0]:
# Remove leading and trailing spaces from column names
clean_columns = [c.strip() for c in bls_df.columns]

# Check for duplicate names after cleaning
assert len(clean_columns) == len(set(clean_columns)), \
    "Duplicate column names found after cleaning"

# Rename columns
bls_df = bls_df.toDF(*clean_columns)

print("Cleaned columns:", bls_df.columns)

In [0]:
from pyspark.sql import functions as F

duplicate_df = (
    bls_df
    .groupBy("series_id", "year", "period")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate key groups:", duplicate_df.count())

display(duplicate_df.limit(20))

In [0]:
print("Column names:")
for c in bls_df.columns:
    print(repr(c))

print("\nSchema:")
bls_df.printSchema()

In [0]:
from pyspark.sql import functions as F

typed_df = (
    bls_df
    .withColumn("year_int", F.col("year").cast("int"))
    .withColumn(
        "value_decimal",
        F.regexp_replace(F.col("value"), ",", "").cast("double")
    )
)

typed_df.select(
    F.count("*").alias("total_records"),
    F.sum(F.when(F.col("year_int").isNull(), 1).otherwise(0))
        .alias("invalid_years"),
    F.sum(F.when(F.col("value_decimal").isNull(), 1).otherwise(0))
        .alias("invalid_values")
).show()

In [0]:
SERIES_PATH = (
    "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.series"
)

series_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "false")
    .csv(SERIES_PATH)
)

# Normalize column names
series_df = series_df.toDF(
    *[c.strip() for c in series_df.columns]
)

print("Series metadata columns:")
print(series_df.columns)

print("Total series:", series_df.count())

display(series_df.limit(10))

In [0]:
# Inspect target series metadata
display(
    series_df
    .filter(F.col("series_id") == "PRS30006032")
)

# Check available Q01 observations
target_q01_df = (
    bls_df
    .filter(
        (F.col("series_id") == "PRS30006032") &
        (F.col("period") == "Q01")
    )
    .orderBy("year")
)

print("Q01 observation count:", target_q01_df.count())

display(target_q01_df.limit(20))

In [0]:
# Search metadata for IDs containing the target digits
display(
    series_df
    .filter(F.col("series_id").contains("3006032"))
)

# Search observations for similar IDs
display(
    bls_df
    .filter(F.col("series_id").contains("3006032"))
    .select("series_id")
    .distinct()
)

# Check whether the exact requested ID exists
print(
    "Exact ID in metadata:",
    series_df.filter(
        F.col("series_id") == "PRS30006032"
    ).count()
)

print(
    "Exact ID in observations:",
    bls_df.filter(
        F.col("series_id") == "PRS30006032"
    ).count()
)

In [0]:
# Correct target series ID
TARGET_ID = "PRS30006032"

print("Metadata matches:")
display(
    series_df.filter(F.col("series_id") == TARGET_ID)
)

print("Observation count:")
print(
    bls_df.filter(F.col("series_id") == TARGET_ID).count()
)

print("Available periods by year:")
display(
    bls_df
    .filter(F.col("series_id") == TARGET_ID)
    .groupBy("year", "period")
    .count()
    .orderBy("year", "period")
    .limit(30)
)

In [0]:
# Inspect IDs after trimming whitespace
series_clean = series_df.withColumn(
    "series_id_clean",
    F.trim(F.col("series_id"))
)

bls_clean = bls_df.withColumn(
    "series_id_clean",
    F.trim(F.col("series_id"))
)

print("Metadata IDs around target:")
display(
    series_clean
    .filter(F.col("series_id_clean").startswith("PRS300060"))
    .select("series_id", "series_id_clean")
    .orderBy("series_id_clean")
)

print("Observation IDs around target:")
display(
    bls_clean
    .filter(F.col("series_id_clean").startswith("PRS300060"))
    .select("series_id_clean")
    .distinct()
    .orderBy("series_id_clean")
)

In [0]:
TARGET_ID = "PRS30006032"

# Search using normalized IDs
target_metadata = (
    series_df
    .filter(F.trim(F.col("series_id")) == TARGET_ID)
)

target_observations = (
    bls_df
    .filter(F.trim(F.col("series_id")) == TARGET_ID)
)

print("Metadata count:", target_metadata.count())
print("Observation count:", target_observations.count())

display(target_metadata)

display(
    target_observations
    .groupBy("year", "period")
    .count()
    .orderBy("year", "period")
)

In [0]:
target_q01 = (
    target_observations
    .filter(F.col("period") == "Q01")
    .withColumn("year_int", F.col("year").cast("int"))
    .withColumn("value_num", F.col("value").cast("double"))
    .orderBy("year_int")
)

print("Q01 record count:", target_q01.count())

display(target_q01)

In [0]:
MEASURE_PATH = (
    "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.measure"
)

measure_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "false")
    .csv(MEASURE_PATH)
)

measure_df = measure_df.toDF(
    *[c.strip() for c in measure_df.columns]
)

print("Measure columns:", measure_df.columns)

display(measure_df)

In [0]:
PERIOD_PATH = (
    "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.period"
)

period_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "false")
    .csv(PERIOD_PATH)
)

period_df = period_df.toDF(
    *[c.strip() for c in period_df.columns]
)

print("Period columns:", period_df.columns)

display(period_df)

In [0]:
import requests

BLS_URL = "https://download.bls.gov/pub/time.series/pr/"

# Replace with your genuine contact email
headers = {
    "User-Agent": "RearcDataQuest/1.0 (contact: YOUR_EMAIL)"
}

response = requests.get(
    BLS_URL,
    headers=headers,
    timeout=30
)

print("HTTP status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response preview:", response.text[:500])

In [0]:
from html.parser import HTMLParser
from urllib.parse import urljoin, urlparse
import os

class BLSFileParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            href = dict(attrs).get("href")
            if href:
                self.links.append(href)


parser = BLSFileParser()
parser.feed(response.text)

bls_files = sorted({
    os.path.basename(urlparse(link).path)
    for link in parser.links
    if urlparse(link).path.startswith("/pub/time.series/pr/")
    and os.path.basename(urlparse(link).path).startswith("pr.")
})

print("Discovered BLS files:", len(bls_files))

for filename in bls_files:
    print(filename)

In [0]:
import hashlib
from pathlib import Path

VOLUME_PATH = "/Volumes/workspace/rearc/rearc_landing"
BLS_RAW_PATH = f"{VOLUME_PATH}/raw/bls"

test_file = "pr.class"
test_url = urljoin(BLS_URL, test_file)
test_path = f"{BLS_RAW_PATH}/{test_file}"

download_response = requests.get(
    test_url,
    headers=headers,
    timeout=60
)

download_response.raise_for_status()

dbutils.fs.put(
    test_path,
    download_response.text,
    overwrite=True
)

checksum = hashlib.sha256(
    download_response.content
).hexdigest()

print("Downloaded:", test_file)
print("HTTP status:", download_response.status_code)
print("File size:", len(download_response.content), "bytes")
print("SHA256:", checksum)
print("Saved to:", test_path)

In [0]:
from html.parser import HTMLParser
from urllib.parse import urljoin
import hashlib
import os
import requests

class BLSFileParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.files = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            href = dict(attrs).get("href", "")
            filename = href.rstrip("/").split("/")[-1]

            if filename.startswith("pr.") and "/" not in filename:
                self.files.append(filename)


# Discover files dynamically
response = requests.get(BLS_URL, headers=headers, timeout=60)
response.raise_for_status()

parser = BLSFileParser()
parser.feed(response.text)

bls_files = sorted(set(parser.files))

print("Files discovered:", len(bls_files))

# Download every discovered file
download_manifest = []

for filename in bls_files:
    file_url = urljoin(BLS_URL, filename)

    file_response = requests.get(
        file_url,
        headers=headers,
        timeout=120
    )
    file_response.raise_for_status()

    destination = f"{BLS_RAW_PATH}/{filename}"

    with open(destination, "wb") as file:
        file.write(file_response.content)

    checksum = hashlib.sha256(
        file_response.content
    ).hexdigest()

    download_manifest.append({
        "filename": filename,
        "size_bytes": len(file_response.content),
        "sha256": checksum
    })

    print("Downloaded:", filename, "| Bytes:", len(file_response.content))

print("Total downloaded:", len(download_manifest))

In [0]:
import os
import json
import hashlib
import requests

manifest_path = f"{VOLUME_PATH}/raw/bls/manifest.json"

# Load the previous manifest, if it exists
if os.path.exists(manifest_path):
    with open(manifest_path, "r") as f:
        old_manifest = json.load(f)
else:
    old_manifest = {"files": {}}

old_files = old_manifest["files"]
new_files = {}

for filename in bls_files:
    response = requests.get(
        f"{BLS_URL}{filename}",
        headers=headers,
        timeout=120
    )
    response.raise_for_status()

    content = response.content
    checksum = hashlib.sha256(content).hexdigest()

    previous = old_files.get(filename)

    if previous and previous["sha256"] == checksum:
        print("UNCHANGED:", filename)
    else:
        destination = f"{BLS_RAW_PATH}/{filename}"
        temp_path = destination + ".tmp"

        with open(temp_path, "wb") as f:
            f.write(content)

        os.replace(temp_path, destination)
        print("NEW OR UPDATED:", filename)

    new_files[filename] = {
        "sha256": checksum,
        "size_bytes": len(content)
    }

removed_files = sorted(set(old_files) - set(new_files))

# Persist the new manifest
manifest = {
    "files": new_files,
    "removed_files": removed_files
}

temp_manifest = manifest_path + ".tmp"

with open(temp_manifest, "w") as f:
    json.dump(manifest, f, indent=2)

os.replace(temp_manifest, manifest_path)

print("Total files:", len(new_files))
print("Removed from source:", removed_files)
print("Manifest saved:", manifest_path)

In [0]:
import requests
import json
import hashlib
import os

DATAUSA_URL = (
    "https://honolulu-api.datausa.io/tesseract/data.jsonrecords"
    "?cube=acs_yg_total_population_1"
    "&drilldowns=Year%2CNation"
    "&locale=en"
    "&measures=Population"
)

DATAUSA_PATH = f"{VOLUME_PATH}/raw/datausa"
os.makedirs(DATAUSA_PATH, exist_ok=True)

response = requests.get(
    DATAUSA_URL,
    headers={"User-Agent": "DataEngineeringAssessment/1.0"},
    timeout=60
)

response.raise_for_status()

population_json = response.json()

output_path = f"{DATAUSA_PATH}/population.json"

with open(output_path, "w") as f:
    json.dump(population_json, f, indent=2)

print("HTTP status:", response.status_code)
print("Records:", len(population_json.get("data", [])))
print("Saved:", output_path)
print("Top-level keys:", list(population_json.keys()))

In [0]:
# Inspect DataUSA JSON structure

print("Columns:")
for column in population_json["columns"]:
    print(column)

print("\nFirst 3 records:")
for record in population_json["data"][:3]:
    print(record)

print("\nTotal records:", len(population_json["data"]))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

# Define an explicit schema
population_schema = StructType([
    StructField("Nation ID", StringType(), True),
    StructField("Nation", StringType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Population", DoubleType(), True)
])

# Create Spark DataFrame
population_bronze_df = spark.createDataFrame(
    population_json["data"],
    schema=population_schema
)

# Standardise column names and add ingestion metadata
population_bronze_df = (
    population_bronze_df
    .withColumnRenamed("Nation ID", "nation_id")
    .withColumnRenamed("Nation", "nation")
    .withColumnRenamed("Year", "year")
    .withColumnRenamed("Population", "population")
    .withColumn("source_system", F.lit("DataUSA"))
    .withColumn("ingested_at", F.current_timestamp())
)

print("Bronze record count:", population_bronze_df.count())

population_bronze_df.printSchema()

display(population_bronze_df.orderBy("year"))